# Use case: Climate Digital Twin generation 2 on a small area 

Earth Data Hub offers an innovative and super-efficient way to access data.

Here we present how to best access the service in the simplest use case.

## Setup the environment

**If you haven't done it already, follow the [Getting started notebook](./00-getting-started.ipynb) to setup your environment and DestinE credentials.**

## Download 30 years of an Climate Digital Twin generation 2 variable on a small area

Our use case is to compute various climatological averages of one ERA5 variable over a long period of time, e.g. the surface temperature over 30 years. Note that we will be using the ERA5 single levels dataset that has more than 100 variable and every variable is almost 3TB in size. 

The best practice for downloading data form the Earth Data Hub comprise the following steps:
1. open the dataset using the code snippet found on the [ERA5 dataset page](https://earthdatahub.destine.eu/collections/era5/datasets/reanalysis-era5-single-levels)
2. select the variable
3. select the area of interest (optionally aligning it on chunk boundaries)
4. select the time interval of interest
5. download the data to memory (with `.persist()` or `.compute()`)
6. save the data or compute the result in memory

### Open the dataset

The following assumes you set up the EDH Personal Access Token in your _netrc_ file.

In [ ]:
import xarray as xr

xr.set_options(keep_attrs=True)

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning,
)

In [ ]:
experiments = ["hist", "SSP3-7.0"]
models = ["ICON", "IFS-NEMO", "IFS-FESOM"]

In [ ]:
climate_dt_ds = {}

for experiment in experiments:
    climate_dt_ds[experiment] = {}
    for model in models:
        print(f"opening {experiment=} {model=}")
        climate_dt_ds[experiment][model] = xr.open_dataset(
            f"https://api.earthdatahub.destine.eu/climate-dt-2/{model}-{experiment}-sfc-hourly-standard-v0.zarr",
            storage_options={"client_kwargs": {"trust_env": True}},
            chunks={},
            engine="zarr",
            zarr_format=3
        )

climate_dt_ds["hist"]["ICON"]

### Select and prepare the variable

Note that all operation are lazy, that is are not applied to the whole 3TB of data, but are just recorded for later use. Download and computations are only done when requested.

In [ ]:
t2m_world = {}

for experiment in experiments:
    t2m_world[experiment] = {}
    for model in models:
        t2m = climate_dt_ds[experiment][model].t2m - 273.15
        t2m.attrs["units"] = "°C"
        t2m.attrs["long_name"] = "surface temperature"
        t2m_world[experiment][model] = t2m

t2m_world["hist"]["ICON"]

### Select the area of interest

For example we are interested in the area of the Alps.

This a very convenient example as:
* it is a large enough area to be more interesting than a time-series of a single point,
* it is small enough to fit a single Zarr chunk, so the notebook can be run even with a slow internet connection,
* when plotted it is easy to identify even without adding coastlines and country borders that will make the notebook more complex.

In [ ]:
aoi_selection = {
    "latitude": slice(45, 51),
    "longitude": slice(5, 15),
}

t2m_aoi_toi = {}

for experiment in experiments:
    t2m_aoi_toi[experiment] = {}
    for model in models:
        t2m_aoi_toi[experiment][model] = t2m_world[experiment][model].sel(**aoi_selection)

t2m_aoi_toi["hist"]["ICON"]

Note that the data size is now much smaller, we went from 3TB for global data to 3GB of our small area of interest.

We plot the map of the temperature at one time to double check that the selection is correct:

In [ ]:
t2m_aoi_toi["hist"]["ICON"].sel(time="2010-01-01T00:00:00").plot()

### Optional: align the area of interest on chunk boundaries

This a pro move and you can skip it.

When accessing the data in Zarr you always download whole chunks, even if you are only interested in part of them. In the case of ERA5 single level the spatial chunks are `(64, 64)` in size and even if you can read above that your DataArray in only 2GB of data you are most probably downloading more data and then throwing a part of it away.

You can use the following (not very nice) code to grow your area of interest to the boundaries of the Zarr chunks, so you use all the data you download. Note that now you also have the size of the data that would be downloaded is it was not compressed (the data is in fact compressed so actual download is smaller).

In [ ]:
def align_indexer(step, indexer):
    if not isinstance(indexer, slice):
        return indexer
    assert indexer.step is None
    start = indexer.start // step * step if indexer.start else indexer.start
    stop = ((indexer.stop - 1) // step + 1) * step if indexer.stop else indexer.stop
    return slice(start, stop)


query_results = xr.core.indexing.map_index_queries(
    t2m_world["hist"]["ICON"], indexers=aoi_selection, method=None, tolerance=None
)
print(query_results.dim_indexers)

aoi_iselection = {
    dim: align_indexer(64, query_results.dim_indexers[dim])
    for dim in query_results.dim_indexers
}
print(aoi_iselection)

for experiment in experiments:
    for model in models:
        t2m_aoi_toi[experiment][model] = t2m_world[experiment][model].isel(**aoi_iselection)

t2m_aoi_toi["hist"]["ICON"]

So, the full time-series for a whole chunk is really 11.5GB, not 2GB.

Let's have a look at the area of the whole chunk.

In [ ]:
t2m_aoi_toi["hist"]["ICON"].sel(time="2010-01-01T00:00:00").plot()

Great! A single 64x64 chunk gets a big part of central Europe.

### Download the data to memory

Finally we are ready to download only the data that we are interested in, in memory.

The best practice is to call the `.compute()` method load the data into a numpy array in memory.

**This operation is the slow one. It takes up to 20 minutes on a 8 Mbps connection.**

It depends on the download speed of your internet connection and on the load on the Earth Data Hub. The closer you are to the data the faster it is, and this is one of the reason the EDH is best suited to be used from within the DestinE platform.

In [ ]:
%%time

t2m_aoi_toi_data = {}

for experiment in experiments:
    t2m_aoi_toi_data[experiment] = {}
    for model in models:
        print(f"downloading {experiment=} {model=}")
        t2m_aoi_toi_data[experiment][model] = t2m_aoi_toi[experiment][model].compute()

t2m_aoi_toi_data["hist"]["ICON"]

#### Monthly climatology

In [ ]:
%%time

t2m_aoi_toi_month_mean = t2m_aoi_toi_data_C.groupby("time.month").mean()
t2m_aoi_toi_month_mean

In [ ]:
t2m_aoi_toi_month_mean.plot(vmin=-30, col="month", col_wrap=3)

#### Daily climatological mean and quintile extremes

In [ ]:
%%time

t2m_aoi_toi_doy_mean = t2m_aoi_toi_data_C.groupby("time.dayofyear").mean()
t2m_aoi_toi_doy_mean = t2m_aoi_toi_doy_mean.sel(dayofyear=slice(0, 365))
t2m_aoi_toi_doy_mean

In [ ]:
point = {
    "latitude": 46.5,
    "longitude": 11.8,
    "method": "nearest",
}

t2m_point_doy_mean = t2m_aoi_toi_doy_mean.sel(**point)

t2m_point_doy_mean.plot()

In [ ]:
%%time

t2m_aoi_toi_doy_quantile = t2m_aoi_toi_data_C.groupby("time.dayofyear").quantile(
    [0.9, 0.5, 0.1]
)
t2m_aoi_toi_doy_quantile = t2m_aoi_toi_doy_quantile.sel(dayofyear=slice(0, 365))
t2m_aoi_toi_doy_quantile

In [ ]:
from matplotlib import pyplot as plt

_, ax = plt.subplots()

t2m_point_doy_quantile = t2m_aoi_toi_doy_quantile.sel(**point)

ax.fill_between(
    t2m_point_doy_quantile.dayofyear,
    t2m_point_doy_quantile.sel(quantile=0.9),
    t2m_point_doy_quantile.sel(quantile=0.1),
    alpha=0.3,
)

t2m_point_doy_quantile.sel(quantile=0.5).plot(ax=ax, alpha=0.5)
t2m_point_doy_mean.plot(ax=ax)

In [ ]:
t2m_ssp3_70_aoi_toi_doy_quantile = (cdt_ifs_nemo_ssp3_70.t2m.sel(time=slice("2048", "2049")) - 273.15).groupby("time.dayofyear").quantile(
    [0.9, 0.5, 0.1]
)
t2m_ssp3_70_aoi_toi_doy_quantile = t2m_ssp3_70_aoi_toi_doy_quantile.sel(dayofyear=slice(0, 365)).compute()
t2m_ssp3_70_aoi_toi_doy_quantile

In [ ]:
_, ax = plt.subplots()

# t2m_ssp3_70_aoi_toi_doy_quantile = t2m_ssp3_70_aoi_toi_doy_quantile.sel(**point)

ax.fill_between(
    t2m_point_doy_quantile.dayofyear,
    t2m_point_doy_quantile.sel(quantile=0.9),
    t2m_point_doy_quantile.sel(quantile=0.1),
    alpha=0.5,
)

ax.fill_between(
    t2m_ssp3_70_aoi_toi_doy_quantile.dayofyear,
    t2m_ssp3_70_aoi_toi_doy_quantile.sel(quantile=0.9),
    t2m_ssp3_70_aoi_toi_doy_quantile.sel(quantile=0.1),
    alpha=0.3,
)

t2m_point_doy_quantile.sel(quantile=0.5).plot(ax=ax, alpha=0.5)
t2m_point_doy_mean.plot(ax=ax)